## 🔍 AIC 2026 — Thử nghiệm Vector Search Trực Quan

Notebook này cho phép bạn gõ một câu query văn bản, mã hoá thành vector 512d qua **LoRA-CLIP** (tự động nạp `lora_weights.pt` nếu có) và truy vấn trực tiếp lên **FAISS Local Index**. Kết quả sẽ hiển thị dưới dạng bảng Markdown trực quan.

### 1. Khởi tạo & Mã hóa Query với LoRA-CLIP

In [ ]:
import numpy as np
from backend.embedding.clip_encoder import encode_text_raw
from backend.config import USE_REMOTE_VECTOR_DB, QDRANT_HOST, QDRANT_PORT, QDRANT_COLLECTION_NAME

# Nhập câu truy vấn tìm kiếm
query_text = "a photo of a tree"

# Mã hóa text sang vector 512d (LoRA-CLIP được tự động áp dụng nếu lora_weights.pt tồn tại)
query_vector = encode_text_raw(query_text)
print(f"Text Query: '{query_text}'")
print(f"Vector Dim: {query_vector.shape}, Norm: {np.linalg.norm(query_vector):.4f}")

### 2. Tìm kiếm similarity trên FAISS Local Index

In [ ]:
import faiss
import json
from backend.config import FAISS_INDEX_PATH, FAISS_METADATA_PATH

index = faiss.read_index(str(FAISS_INDEX_PATH))
with open(FAISS_METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
print('Đã load FAISS Index thành công!')


## 🔍 Tìm kiếm Nâng Cao (MMR & Rocchio Feedback)
Tìm kiếm với **MMR (Maximal Marginal Relevance)** để cân bằng giữa độ chính xác (Relevance) và độ đa dạng (Diversity).
Đồng thời, áp dụng **Rocchio Relevance Feedback** để cập nhật lại Vector truy vấn sau khi người dùng đánh giá kết quả đúng/sai.

In [ ]:
from backend.embedding.search_algorithms import mmr_search, rocchio_feedback
import numpy as np
from pathlib import Path
from PIL import Image
import faiss

query_super = vec.reshape(1, -1).astype(np.float32)
faiss.normalize_L2(query_super)

top_k = 10

# 1. Tìm kiếm MMR
scores_mmr, results_mmr = mmr_search(query_super, index, metadata, top_k=top_k, lambda_mult=0.5, fetch_k=50)

# 2. Rocchio Feedback (Giả sử kết quả #1 là Tốt, kết quả cuối cùng là Tệ)
scores_rocchio, results_rocchio = scores_mmr, results_mmr
if len(results_mmr) >= 2:
    rel_vecs = [index.reconstruct(int(results_mmr[0]['id']))]
    non_rel_vecs = [index.reconstruct(int(results_mmr[-1]['id']))]
    new_query = rocchio_feedback(query_super, rel_vecs, non_rel_vecs, alpha=1.0, beta=0.75, gamma=0.15)
    scores_rocchio, results_rocchio = mmr_search(new_query, index, metadata, top_k=top_k, lambda_mult=0.5, fetch_k=50)

artifact_dir = Path('search_results').resolve()
artifact_dir.mkdir(parents=True, exist_ok=True)

def render_table(title, scores, results):
    content = f"## {title}\n\n"
    content += f"| Rank | Score | Video | Frame | Time | Image |\n"
    content += f"| :---: | :---: | :---: | :---: | :---: | :---: |\n"
    for i, (score, p) in enumerate(zip(scores, results)):
        vid = p.get('video_id', 'N/A')
        fid = p.get('frame_id', 0)
        pts = p.get('pts_time', 0.0)
        img_path = Path(p.get('path', ''))
        img_artifact = artifact_dir / f"{vid}_{fid}.jpg"
        if img_path.exists():
            Image.open(img_path).save(img_artifact)
        content += f"| **#{i+1:02d}** | `{score:.4f}` | `{vid}` | `{fid}` | `{pts:.1f}s` | ![{vid}_{fid}]({img_artifact.as_uri()}) |\n"
    return content

md_content = f"# Báo Cáo Tìm Kiếm (MMR & Rocchio)\n\n"
md_content += f"- **Query**: `{query_text}`\n\n"
md_content += render_table("1. Tìm kiếm MMR (lambda=0.5)", scores_mmr, results_mmr)
md_content += render_table("2. Tìm kiếm Rocchio Feedback (Cập nhật vector)", scores_rocchio, results_rocchio)

from IPython.display import display, Markdown
display(Markdown(md_content))
